In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import pickle
from catboost import CatBoostRegressor

In [2]:
banners = pd.read_csv("../data/db/banners.csv", parse_dates=["created_at"])
interactions = pd.read_csv("../data/db/banner_interactions.csv", parse_dates=["event_date"])
users = pd.read_csv("../data/db/users.csv")

In [3]:
df = interactions.merge(users, on="user_id", how="left")
df = df.merge(banners, on="banner_id", how="left")

In [5]:
df["target_ctr"] = df["clicks"] / df["impressions"]

In [6]:
df["age_match"] = (
    (df["age"] >= df["target_age_min"]) &
    (df["age"] <= df["target_age_max"])
).astype(int)

df["gender_match"] = (
    (df["target_gender"] == "U") |
    (df["gender"] == df["target_gender"]) |
    (df["gender"] == "U")
).astype(int)

df["interest_match"] = (
    (df["interest_1"] == df["subcategory"]) |
    (df["interest_2"] == df["subcategory"]) |
    (df["interest_3"] == df["subcategory"])
).astype(int)

df["banner_age_days"] = (
    df["event_date"] - df["created_at"]
).dt.days.clip(lower=0)

df["event_dow"] = df["event_date"].dt.dayofweek

In [7]:
df.head()

,event_date,user_id,banner_id,impressions,clicks,ctr,age,gender,city_tier,device_os,...,quality_score,created_at,is_active,landing_page,target_ctr,age_match,gender_match,interest_match,banner_age_days,event_dow
0,2026-01-01,u_00007,b_0082,3,0,0.0,20,M,tier_2,Android,...,0.507,2025-12-30,1,https://example.com/food/snacks/orbit-82,0.0,0,0,0,2,3
1,2026-01-01,u_00009,b_0234,3,0,0.0,39,U,tier_2,iOS,...,0.949,2025-10-19,1,https://example.com/food/delivery/delta-234,0.0,0,1,0,74,3
2,2026-01-01,u_00012,b_0041,4,0,0.0,41,F,tier_1,Windows,...,0.843,2025-12-22,1,https://example.com/travel/hotels/aura-41,0.0,1,0,0,10,3
3,2026-01-01,u_00012,b_0215,5,1,0.2,41,F,tier_1,Windows,...,0.757,2025-12-24,1,https://example.com/entertainment/events/vanta...,0.2,1,1,0,8,3
4,2026-01-01,u_00015,b_0210,5,2,0.4,46,F,tier_3,Android,...,0.559,2025-10-28,1,https://example.com/finance/insurance/vanta-210,0.4,1,1,1,65,3


In [8]:
train_df = df[df["event_date"] < "2026-03-16"].copy()
valid_df = df[df["event_date"] >= "2026-03-16"].copy()

In [9]:
global_ctr = train_df["clicks"].sum() / train_df["impressions"].sum()

In [10]:
def add_smoothed_ctr_feature(train_df, apply_df, group_cols, feature_name, alpha=100):
    agg = train_df.groupby(group_cols)[["clicks", "impressions"]].sum().reset_index()
    agg[feature_name] = (agg["clicks"] + alpha * global_ctr) / (agg["impressions"] + alpha)

    train_df = train_df.merge(agg[group_cols + [feature_name]], on=group_cols, how="left")
    apply_df = apply_df.merge(agg[group_cols + [feature_name]], on=group_cols, how="left")

    train_df[feature_name] = train_df[feature_name].fillna(global_ctr)
    apply_df[feature_name] = apply_df[feature_name].fillna(global_ctr)
    return train_df, apply_df

for cols, name, alpha in [
    (["banner_id"], "banner_ctr_prior", 100),
    (["user_id"], "user_ctr_prior", 100),
    (["subcategory"], "subcategory_ctr_prior", 100),
    (["user_id", "subcategory"], "user_subcat_ctr_prior", 20),
]:
    train_df, valid_df = add_smoothed_ctr_feature(train_df, valid_df, cols, name, alpha)

In [11]:
features = [
    "user_id",
    "banner_id",
    "age",
    "gender",
    "city_tier",
    "device_os",
    "platform",
    "income_band",
    "activity_segment",
    "interest_1",
    "interest_2",
    "interest_3",
    "signup_days_ago",
    "is_premium",
    "brand",
    "category",
    "subcategory",
    "banner_format",
    "campaign_goal",
    "target_gender",
    "target_age_min",
    "target_age_max",
    "cpm_bid",
    "quality_score",
    "is_active",
    "age_match",
    "gender_match",
    "interest_match",
    "banner_age_days",
    "event_dow",
    "banner_ctr_prior",
    "user_ctr_prior",
    "subcategory_ctr_prior",
    "user_subcat_ctr_prior",
]

In [12]:
cat_features = [
    "user_id",
    "banner_id",
    "gender",
    "city_tier",
    "device_os",
    "platform",
    "income_band",
    "activity_segment",
    "interest_1",
    "interest_2",
    "interest_3",
    "brand",
    "category",
    "subcategory",
    "banner_format",
    "campaign_goal",
    "target_gender",
]

In [13]:
# 8. Обучение
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    verbose=100
)

In [14]:
model.fit(
    train_df[features],
    train_df["target_ctr"],
    cat_features=cat_features,
    sample_weight=train_df["impressions"],
    eval_set=(valid_df[features], valid_df["target_ctr"]),
    use_best_model=True
)

0:	learn: 0.0873046	test: 0.1057143	best: 0.1057143 (0)	total: 138ms	remaining: 1m 8s
100:	learn: 0.0402822	test: 0.1112225	best: 0.1057143 (0)	total: 9.02s	remaining: 35.6s
200:	learn: 0.0380256	test: 0.1113364	best: 0.1057143 (0)	total: 18.2s	remaining: 27.1s
300:	learn: 0.0369193	test: 0.1115872	best: 0.1057143 (0)	total: 28.5s	remaining: 18.8s
400:	learn: 0.0362730	test: 0.1117248	best: 0.1057143 (0)	total: 39.9s	remaining: 9.85s
499:	learn: 0.0358718	test: 0.1117866	best: 0.1057143 (0)	total: 52.1s	remaining: 0us

bestTest = 0.1057142919
bestIteration = 0

Shrink model to first 1 iterations.


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [15]:
def recommend_banners_for_user(user_id, users_df, banners_df, model, history_features_df=None, top_k=10):
    user_row = users_df[users_df["user_id"] == user_id].copy()
    active_banners = banners_df[banners_df["is_active"] == 1].copy()

    candidates = active_banners.assign(key=1).merge(
        user_row.assign(key=1), on="key"
    ).drop(columns="key")

    # Текущая дата показа
    today = pd.Timestamp("2026-04-01")
    candidates["event_dow"] = today.dayofweek
    candidates["banner_age_days"] = (today - candidates["created_at"]).dt.days.clip(lower=0)

    candidates["age_match"] = (
        (candidates["age"] >= candidates["target_age_min"]) &
        (candidates["age"] <= candidates["target_age_max"])
    ).astype(int)

    candidates["gender_match"] = (
        (candidates["target_gender"] == "U") |
        (candidates["gender"] == candidates["target_gender"]) |
        (candidates["gender"] == "U")
    ).astype(int)

    candidates["interest_match"] = (
        (candidates["interest_1"] == candidates["subcategory"]) |
        (candidates["interest_2"] == candidates["subcategory"]) |
        (candidates["interest_3"] == candidates["subcategory"])
    ).astype(int)

    # Здесь надо подлить заранее рассчитанные history-features
    # banner_ctr_prior, user_ctr_prior, user_subcat_ctr_prior и т.д.

    candidates["pred_ctr"] = model.predict(candidates[features])
    recs = candidates.sort_values("pred_ctr", ascending=False).head(top_k)

    return recs[["banner_id", "brand", "category", "subcategory", "pred_ctr"]]

In [16]:
recommend_banners_for_user(
    user_id="u_00007",
    users_df=users,
    banners_df=banners,
    model=model
    )

KeyError: "['banner_ctr_prior', 'user_ctr_prior', 'subcategory_ctr_prior', 'user_subcat_ctr_prior'] not in index"